# Task 2: Imitation Learning

## Setup Code / Packages
If you are running the code locally follow the instructions [here](../docs/setup.md).

If you are using Google Colab make sure to run the two cells below to install all dependencies and enable headless rendering. You will also need to upload your xml file to the assets folder.

Task 2 additionally requires PyTorch and MPlib. On Linux or Colab, install them with `pip install torch mplib`. MPlib prebuilt packages do not support macOS.


In [ ]:
import os, sys

# Pick MuJoCo backend
if sys.platform == "darwin":
    os.environ["MUJOCO_GL"] = "glfw"            # macOS
elif sys.platform.startswith("linux"):
    os.environ["MUJOCO_GL"] = "glfw"            # Colab CPU default
else:
    os.environ["MUJOCO_GL"] = "glfw"

# Start a virtual display only on Linux (e.g., Colab)
if sys.platform.startswith("linux"):
    try:
        from pyvirtualdisplay import Display
        Display(visible=0, size=(1280,720)).start()
    except Exception:
        pass  # not installed (fine on macOS)


In [ ]:
import os
os.environ["MUJOCO_GL"] = "glfw"


## Using Gymnasium
No problems are assigned here, but understanding the code APIs/interfaces and functionalities are important.

### Gymnasium Basics
The Gymnasium package is a standardized API for creating and interacting with simulation environments. 

<img src="../assets/media/gym_loop.png" alt="agent-environment loop" width="500"/>

Gymnasium defines the agent-environment loop — a simple interface where an agent takes actions and receives observations, rewards, and termination signals from an environment. This loop forms the foundation for reinforcement learning (RL) and allows us to easily manipulate the simulator.

**Why We Use Gymnasium**

Gymnasium acts as a common interface layer between learning algorithms and simulation environments. It provides:
- A uniform API (reset(), step(), observation, reward, done, info) 
- Integration with RL algorithm libraries (e.g., Stable-Baselines3, CleanRL).
- Rendering and debugging utilities for visualization and evaluation.


In this task, you will use Gymnasium as the primary API to interface with the simulator to :
- Collect trajectory data for imitation learning.
- Evaluate and deploy trained policies.

Read their documentation [here](https://gymnasium.farama.org/).


## Part 0: Creating a Standardized Environment with Gymnasium
In this part, you will implement the `TrajEnv` class to wrap a MuJoCo simulation in a Gymnasium interface.

Your task is to decide which observations are useful as states in a behavior cloning setting. You will also implement the `is_grasped` property to define when the robot has successfully grasped the object and the `terminated` property to specify when the cube is in the bin. The `reset_model` method should randomize initial arm and object positions to encourage generalization.

You should step through Gymnasium functions like `do_simulation` and `set_state` to see how to control the simulator.

In [ ]:
from pathlib import Path

# Works when Jupyter starts in either the repository root or task2/.
_candidates = [Path.cwd() / "assets/descriptions/DropCubeInBinEnv.xml",
              Path.cwd().parent / "assets/descriptions/DropCubeInBinEnv.xml"]
XML_PATH = str(next((p.resolve() for p in _candidates if p.is_file()), _candidates[0].resolve()))
assert Path(XML_PATH).is_file(), f"Missing scene XML: {XML_PATH}"


In [ ]:
from gymnasium.envs.mujoco import MujocoEnv
from gymnasium.spaces import Box
import mujoco
import numpy as np


def _safe_body_id(model, name):
    body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, name)
    return None if body_id < 0 else body_id


class TrajEnv(MujocoEnv):
    metadata = {"render_modes": ["human", "rgb_array", "depth_array"]}

    def __init__(self, xml_file, frame_skip=20, **kwargs):
        model = mujoco.MjModel.from_xml_path(xml_file)
        observation_space = Box(-np.inf, np.inf, shape=(model.nq + model.nv,), dtype=np.float32)
        super().__init__(xml_file, frame_skip, observation_space=observation_space, **kwargs)
        self.panda_id = _safe_body_id(self.model, "link0")
        self.hand_tcp_id = _safe_body_id(self.model, "panda_hand_tcp")
        self.cube_id = _safe_body_id(self.model, "cube")
        self.bin_id = _safe_body_id(self.model, "bin")
        self.left_finger_id = _safe_body_id(self.model, "left_finger")
        self.right_finger_id = _safe_body_id(self.model, "right_finger")
        if any(x is None for x in (self.panda_id, self.hand_tcp_id, self.cube_id, self.bin_id)):
            raise ValueError("Scene must contain link0, panda_hand_tcp, cube, and bin bodies")

    @property
    def is_grasped(self):
        finger_ids = {self.left_finger_id, self.right_finger_id}
        cube_geoms = {i for i in range(self.model.ngeom) if self.model.geom_bodyid[i] == self.cube_id}
        for i in range(self.data.ncon):
            contact = self.data.contact[i]
            b1 = self.model.geom_bodyid[contact.geom1]
            b2 = self.model.geom_bodyid[contact.geom2]
            if (contact.geom1 in cube_geoms and b2 in finger_ids) or (contact.geom2 in cube_geoms and b1 in finger_ids):
                return True
        return False

    @property
    def terminated(self):
        cube = self.data.xpos[self.cube_id]
        bin_pos = self.data.xpos[self.bin_id]
        # Bin interior is 9 cm square; allow cube half-width and a small margin.
        inside_xy = np.all(np.abs(cube[:2] - bin_pos[:2]) <= 0.02)
        inside_z = bin_pos[2] + 0.029 <= cube[2] <= bin_pos[2] + 0.061
        return bool(inside_xy and inside_z)

    def _get_obs(self):
        return np.concatenate((self.data.qpos, self.data.qvel)).astype(np.float32)

    def step(self, action):
        self.do_simulation(np.clip(action, self.action_space.low, self.action_space.high), self.frame_skip)
        terminated = self.terminated
        return self._get_obs(), float(terminated), terminated, False, {"is_grasped": self.is_grasped}

    def reset_model(self):
        qpos = np.array(self.model.keyframe("home").qpos, dtype=np.float64)
        qvel = np.zeros(self.model.nv)
        # Small cube position variation keeps expert planning feasible while adding coverage.
        qpos[9:11] += self.np_random.uniform(-0.01, 0.01, size=2)
        self.set_state(qpos, qvel)
        return self._get_obs()


To understand what is happening in the environment, you can call `env.render()` to visualize it.  
Since we set `render_mode="rgb_array"`, `env.render()` will return an RGB array that can be displayed.  


> **Note:** If the default view of the environment is not ideal, you may need to adjust the camera.  
> You can do this by defining a camera in your Mujoco XML file and then passing its name as a kwarg to the `TrajEnv` constructor via the `camera_name` argument.


If you are on colab you can render the environment like so. If you are running MuJoCo locally you can use `render_mode="human"` to display viewing window.


In [ ]:
from gymnasium.wrappers import TimeLimit
import matplotlib.pyplot as plt

env = TrajEnv(XML_PATH, 20, render_mode='rgb_array', camera_name="fixed")
env = TimeLimit(env, 100)
env.reset()
image = env.render() 
image = image
plt.imshow(image)

With these core functionalities we can build the basic interaction loop that consists of a reset followed by steps until terminated or truncated is True. We can also record a video as well.

In [ ]:
obs, _ = env.reset()
done = False
images = [env.render()]
# keep taking steps until either we terminated or truncate
while not done:
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    images.append(env.render())
    done = terminated or truncated

In [ ]:
from IPython.display import Video
import imageio

frames = images 
frames = [frame.astype(np.uint8) for frame in frames]   

# Write video to file
video_path = "output_video.mp4"
with imageio.get_writer(video_path, fps=20) as writer:
    for frame in frames:
        writer.append_data(frame)

# Display video in notebook
Video(video_path, embed=True)

We will now try and control the robot to grasp the cube, pick it up, and drop it in the wooden bin. 

The following code sets up a motion planning library for you and saves some useful variables. 
- `planner`: The planner objects uses screw motion planning.
- `panda_hand_tcp`: The panda hand tool center point "link", which gives the current pose of the link we are controlling via the motion planning library. This is the point in the middle between the two panda arm grippers.
- `cube`: The cube object, which you can use to acccess its pose.
- `bin`: The bin object.

The motion planner setup will generate a sequence joint position actions for all the joints excluding the gripper. The gripper actions you can manually pick (can range from 0 to 255). 

In [ ]:
import mplib
from pathlib import Path

from gymnasium.wrappers import TimeLimit

env = TrajEnv(XML_PATH, frame_skip=20, render_mode="rgb_array")
env.reset(seed=42)

link_names = [
    "panda_link0", "panda_link1", "panda_link2", "panda_link3", "panda_link4",
    "panda_link5", "panda_link6", "panda_link7", "panda_link8", "panda_hand",
    "panda_hand_tcp", "panda_leftfinger", "panda_rightfinger",
    "panda_leftfinger_pad", "panda_rightfinger_pad",
]
joint_names = [
    "panda_joint1", "panda_joint2", "panda_joint3", "panda_joint4",
    "panda_joint5", "panda_joint6", "panda_joint7", "panda_finger_joint1", "panda_finger_joint2",
]
planner = mplib.Planner(
    urdf=str(Path(XML_PATH).parent / "panda/urdf/panda.urdf"),
    srdf=str(Path(XML_PATH).parent / "panda/urdf/panda.srdf"),
    user_link_names=link_names,
    user_joint_names=joint_names,
    move_group="panda_hand_tcp",
    joint_vel_limits=np.ones(7) * 0.8,
    joint_acc_limits=np.ones(7) * 0.8,
)
planner.set_base_pose(mplib.Pose(env.data.xpos[env.panda_id].copy(), env.data.xquat[env.panda_id].copy()))


**Notes for this task**

The goal is to use the motion planner to pick up the cube and drop it into the bin.

- You will need to access object positions and orientations from the simulator (env.data.xpos, env.data.xquat, etc.).
- The motion planner requires a target pose (7,) for the robot hand (position + quaternion) and the current joint configuration (qpos). Both are expressed in the world frame.
- Plan a trajectory to a consistent grasp pose above the cube, execute it while keeping the gripper open, then move down and close the gripper to pick up the cube.    
- Finally, open the gripper to release the cube into the bin.

The planner may fail if the pose is unreachable, collides with the environment, or the solver does not converge. This is expected—adjust poses or retry if needed.

In [ ]:
from gymnasium.wrappers import RecordVideo
from pathlib import Path

EPISODES = 5
RUN_PLANNER_DEMO = True

def _normalized_action(env, action):
    center = (env.action_space.high + env.action_space.low) / 2
    scale = (env.action_space.high - env.action_space.low) / 2
    return np.clip((action - center) / scale, -1, 1).astype(np.float32)

def _raw_action(env, normalized):
    center = (env.action_space.high + env.action_space.low) / 2
    scale = (env.action_space.high - env.action_space.low) / 2
    return np.clip(normalized * scale + center, env.action_space.low, env.action_space.high)

def pick_cube_solution(env, episode_data=None):
    """Execute an expert approach, grasp, lift, transfer, and release sequence."""
    episode_data = [] if episode_data is None else episode_data
    planner.set_base_pose(mplib.Pose(env.data.xpos[env.panda_id].copy(), env.data.xquat[env.panda_id].copy()))
    cube = env.data.xpos[env.cube_id].copy()
    bin_pos = env.data.xpos[env.bin_id].copy()
    tcp_quat = env.data.xquat[env.hand_tcp_id].copy()

    def move_to(position, gripper=255):
        goal = mplib.Pose(np.asarray(position, dtype=np.float64), tcp_quat)
        result = planner.plan_screw(goal, env.data.qpos[:7].copy(), time_step=0.02)
        if result.get("status") != "Success":
            return False
        for q in result["position"]:
            action = np.zeros(env.action_space.shape, dtype=np.float64)
            action[:7] = q
            action[7] = gripper
            episode_data.append((env._get_obs().copy(), _normalized_action(env, action)))
            _, _, terminated, _, _ = env.step(action)
            if terminated:
                return True
        return True

    def hold_gripper(value, steps):
        arm = env.data.qpos[:7].copy()
        for _ in range(steps):
            action = np.zeros(env.action_space.shape, dtype=np.float64)
            action[:7] = arm
            action[7] = value
            episode_data.append((env._get_obs().copy(), _normalized_action(env, action)))
            _, _, terminated, _, _ = env.step(action)
            if terminated:
                return True
        return False

    # Gripper is open at 255 and closed at 0 in the included Panda actuator.
    stages = [
        (cube + [0, 0, 0.12], 255),
        (cube + [0, 0, 0.035], 255),
    ]
    for target, grip in stages:
        if not move_to(target, grip):
            return False
    hold_gripper(0, 20)
    if not env.is_grasped:
        return False
    if not move_to(cube + [0, 0, 0.14], 0):
        return False
    if not move_to(bin_pos + [0, 0, 0.16], 0):
        return False
    hold_gripper(255, 35)
    return env.terminated

if RUN_PLANNER_DEMO:
    demo_env = RecordVideo(env, "videos", episode_trigger=lambda _: True, name_prefix="planner")
    successes = 0
    for i in range(EPISODES):
        demo_env.reset(seed=i)
        successes += bool(pick_cube_solution(demo_env))
        print(f"Episode {i + 1}: success={demo_env.unwrapped.terminated}")
    print(f"Planner success rate: {successes / EPISODES:.1%}")
    demo_env.close()


In [ ]:
from IPython.display import Video
Video("videos/planner-episode-0.mp4", embed=True, width=320) # Watch replay

## Part 1: Collect your Imitation Learning Dataset


Now that you’ve used the motion planner to complete a single pick-and-place sequence, the next step is to repeat this process many times to collect a dataset for imitation learning.

In imitation learning, the dataset is made of trajectories—sequences of state-action pairs showing how an expert (in this case, the motion planner) solves the task. Each trajectory will later be used to train a policy that imitates this behavior.

>**Tip**: Standardizing the orientation and approach of the gripper will help the robot pick up the cube in the same way each time, resulting in a more reliable model.


In [ ]:
import pickle
from pathlib import Path

EPISODES = 50  # Start small; increase only after the planner success rate is useful.
env = TrajEnv(XML_PATH, frame_skip=20)
trajectories = []
success_count = 0
for i in range(EPISODES):
    env.reset(seed=i)
    episode_data = []
    if pick_cube_solution(env, episode_data):
        trajectories.append(episode_data)
        success_count += 1
    if (i + 1) % 10 == 0:
        print(f"Episodes {i + 1}/{EPISODES}; successes={success_count}")

with open("pick_place_dataset.pkl", "wb") as f:
    pickle.dump(trajectories, f)
print(f"Success rate: {success_count / EPISODES:.1%}; successful trajectories: {len(trajectories)}")
env.close()


## Part 2: Create Behavior Cloning Dataset

Using the trajectory data you have collected in the previous task, complete the `TrajectoryDataset` class to feed your model. 

>**NOTE**: The model learns to map states to actions, so the order of the data is not important. You can flatten all episodes and sample from any step freely.

In [ ]:
from torch.utils.data import Dataset
import torch

class TrajectoryDataset(Dataset):
    def __init__(self, trajectories):
        pairs = [pair for trajectory in trajectories for pair in trajectory]
        if not pairs:
            raise ValueError("Dataset is empty; collect successful planner trajectories first")
        observations, actions = zip(*pairs)
        self.observations = torch.as_tensor(np.asarray(observations), dtype=torch.float32)
        self.actions = torch.as_tensor(np.asarray(actions), dtype=torch.float32)

    def __len__(self):
        return len(self.observations)

    def __getitem__(self, idx):
        return self.observations[idx], self.actions[idx]

with open("pick_place_dataset.pkl", "rb") as f:
    dataset = TrajectoryDataset(pickle.load(f))
print(f"Training samples: {len(dataset)}; obs={tuple(dataset.observations.shape[1:])}; actions={tuple(dataset.actions.shape[1:])}")


## Part 3: Define the Policy Network

In this step, you will implement a behavior cloning policy network. The goal of this network is to learn to map observations (states) to actions by imitating the trajectories you collected earlier. 

**NOTE**: See [tips](tips.md) for guidance on designing your network architecture and BC overview.

In [ ]:
import torch
import torch.nn as nn

class Actor(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(obs_dim, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh())
        self.mean = nn.Linear(256, action_dim)
        self.log_std = nn.Parameter(torch.full((action_dim,), -1.0))

    def forward(self, x):
        hidden = self.trunk(x)
        mean = self.mean(hidden)
        return mean, self.log_std.clamp(-5, 2).expand_as(mean)


## Part 4: Training Loop
You will train the neural network policy to imitate expert behavior.

We’ve provided an `eval_policy` function to test your actor in simulation. It runs several episodes and reports a success rate, so you can track how well your policy performs as training progresses.

In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn

def eval_policy(eval_env: gym.Env, actor: nn.Module, num_episodes: int = 5):
    device = next(actor.parameters()).device
    base_env = eval_env.unwrapped
    success = 0
    for ep in range(num_episodes):
        obs, _ = eval_env.reset(seed=1000 + ep)
        terminated = truncated = False
        while not (terminated or truncated):
            with torch.no_grad():
                normalized, _ = actor(torch.as_tensor(obs, dtype=torch.float32, device=device))
            action = _raw_action(base_env, normalized.cpu().numpy())
            obs, _, terminated, truncated, _ = eval_env.step(action)
        success += bool(terminated)
        print(f"Episode {ep + 1}: success={bool(terminated)}")
    return success / num_episodes


In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)
actor = Actor(dataset.observations.shape[1], dataset.actions.shape[1]).to(device)
train_env = TrajEnv(XML_PATH, 20)
eval_env = TimeLimit(TrajEnv(XML_PATH, 20), max_episode_steps=100)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)
optimizer = torch.optim.Adam(actor.parameters(), lr=3e-4)

def nll_loss(actor, obs, expert_actions):
    mean, log_std = actor(obs)
    return -torch.distributions.Normal(mean, log_std.exp()).log_prob(expert_actions).sum(-1).mean()

best_success = -1.0
for epoch in range(50):
    actor.train()
    losses = []
    for obs, actions in dataloader:
        loss = nll_loss(actor, obs.to(device), actions.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if (epoch + 1) % 5 == 0:
        actor.eval()
        success = eval_policy(eval_env, actor, num_episodes=2)
        print(f"Epoch {epoch + 1}: loss={np.mean(losses):.4f}, success={success:.1%}")
        if success > best_success:
            best_success = success
            torch.save(actor.state_dict(), "best_policy.pth")
actor.eval()
train_env.close()
eval_env.close()


## Part 5: Evaluate in Simulator
Time to evaluate your model. You may notice that it doesn’t succeed very often and may behave unpredictably or erratically. 

Can you think about why this might happen?

In [ ]:
import torch
from gymnasium.wrappers import RecordVideo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
actor = Actor(dataset.observations.shape[1], dataset.actions.shape[1]).to(device)
actor.load_state_dict(torch.load("best_policy.pth", map_location=device))
actor.eval()
eval_env = RecordVideo(TimeLimit(TrajEnv(XML_PATH, 20, render_mode="rgb_array"), 100), "videos", episode_trigger=lambda _: True, name_prefix="bc-model")
eval_policy(eval_env, actor, 5)
eval_env.close()


In [ ]:
from IPython.display import Video
Video("videos/bc-model-episode-0.mp4", embed=True, width=320) # Watch our replay